# ML Baseline And Fusion Search

This notebook runs fast strawberry-only experiments before fine-tuning the deep models. It compares sequence length and fusion style using image summary features plus temperature/humidity.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

LAB_DIR = Path.cwd()
if LAB_DIR.name != 'strawberry':
    LAB_DIR = Path('notebooks/strawberry').resolve()
sys.path.insert(0, str(LAB_DIR))
import lab_utils as lab


## Run Sweep

This may take a few minutes because it reads strawberry images and fits several sklearn regressors.


In [ ]:
results = lab.run_ml_baseline_sweep(seq_lens=(3, 5, 8, 10))
results.head(20)


In [ ]:
results.sort_values(['val_mae', 'test_mae']).head(10)


In [ ]:
rf = results[results['model'] == 'random_forest'].copy()
fig, ax = plt.subplots(figsize=(10, 5))
for fusion, group in rf.groupby('fusion'):
    ax.plot(group['seq_len'], group['test_mae'], marker='o', label=fusion)
ax.axhline(32.02, color='black', linestyle='--', linewidth=1, label='Model C reported MAE')
ax.set_xlabel('seq_len')
ax.set_ylabel('Test MAE (hours)')
ax.set_title('RandomForest fusion sweep')
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()


In [ ]:
best = results.sort_values(['val_mae', 'test_mae']).iloc[0].to_dict()
best


## How To Interpret

Use validation MAE for config selection, then treat test metrics as the final held-out check. If image-only beats early fusion on test but early fusion wins validation, the safest deep-model transfer is image-dominant fusion with environment as a side branch, not raw env concatenation as the main signal.
